
# Load YAMNet
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

# Load audio
waveform, sr = librosa.load('file path.wav', sr=16000)

# extract embedding
scores, embeddings, spectrogram = yamnet(waveform)
embeddings = embeddings.numpy()  # (frame, 1024)

audio inference：
# （if you've already have embeddings）

# split into small segments
from utils import create_sequences

X_test_seq, _ = create_sequences(embeddings, np.zeros(len(embeddings)), sequence_length=60)

# if audio is too short，X_test_seq might be empty，so chack first
if len(X_test_seq) == 0:
    print("if audio is too short，not enough to form 60 frames，plz change a longer one。")
else:
    # pridict by the model
    y_pred_probs = model.predict(X_test_seq)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # get result
    from scipy.stats import mode
    final_prediction = mode(y_pred, keepdims=False).mode

    # print the name of each audio 
    instrument_classes = {
        0: 'cello',
        1: 'clarinet',
        2: 'flute',
        3: 'acoustic guitar',
        4: 'electric guitar',
        5: 'organ',
        6: 'piano',
        7: 'saxophone',
        8: 'trumpet',
        9: 'violin',
        10: 'vocal'
    }

    print("Predicted instrument:", instrument_classes[final_prediction])

 加载训练好的模型：
 from tensorflow.keras.models import load_model
from tcn import TCN  # 如果用了TCN层需要import

model = load_model('checkpoints/tcn_snr-5_model.h5', custom_objects={'TCN': m.TCN})

In [120]:
import librosa
import tensorflow_hub as hub
import tensorflow as tf  

In [122]:
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

In [123]:
from tensorflow.keras.models import load_model
from tcn import TCN

In [158]:
model = load_model('checkpoints/tcn_snr-5_model.h5', custom_objects={'TCN': m.TCN})

In [160]:
audio_dir = os.path.expanduser('~/Desktop/final-main/分析')
file_list = sorted([f for f in os.listdir(audio_dir) if f.endswith('.wav')])

In [164]:
waveforms = []
srs = []
all_embeddings = []

for filename in file_list:
    file_path = os.path.join(audio_dir, filename)
    waveform, sr = librosa.load(file_path, sr=16000)  # 强制采样率16kHz
    waveforms.append(waveform)
    srs.append(sr)

    scores, embeddings, spectrogram = yamnet(waveform)
    embedding = embeddings.numpy()  # (帧数, 1024)
    all_embeddings.append(embedding)
    print(f"Processed {filename}: embedding shape {embedding.shape}")

# 填充所有的 embeddings 
max_frames = 60  
X_test_seq = pad_sequences(all_embeddings, maxlen=max_frames, dtype='float32', padding='post', truncating='post')

print(f"X_test_seq shape: {X_test_seq.shape}")

Processed 1-cello.wav: embedding shape (66, 1024)
Processed 10 clarinet.wav: embedding shape (63, 1024)
Processed 11 electric guitar.wav: embedding shape (63, 1024)
Processed 12 flute.wav: embedding shape (63, 1024)
Processed 13 human vocie.wav: embedding shape (63, 1024)
Processed 14 human voice-1.wav: embedding shape (63, 1024)
Processed 15 human voice.wav: embedding shape (63, 1024)
Processed 16 orgen.wav: embedding shape (63, 1024)
Processed 17 pinao.wav: embedding shape (63, 1024)
Processed 17 saxphone.wav: embedding shape (63, 1024)
Processed 19 trumpet.wav: embedding shape (63, 1024)
Processed 2-acoustic guitar.wav: embedding shape (66, 1024)
Processed 20 violin.wav: embedding shape (63, 1024)
Processed 3-oboe.wav: embedding shape (66, 1024)
Processed 4-organ.wav: embedding shape (66, 1024)
Processed 5-piano.wav: embedding shape (66, 1024)
Processed 6-trumpet.wav: embedding shape (66, 1024)
Processed 7-violin.wav: embedding shape (66, 1024)
Processed 8 acoustic guitar.wav: embed

In [166]:
y_pred_probs = model.predict(X_test_seq)
y_pred = np.argmax(y_pred_probs, axis=1)

# 乐器类别
instrument_classes = {
    0: 'cello',
    1: 'clarinet',
    2: 'flute',
    3: 'acoustic guitar',
    4: 'electric guitar',
    5: 'organ',
    6: 'piano',
    7: 'saxophone',
    8: 'trumpet',
    9: 'violin',
    10: 'vocal'
}

for idx, (filename, pred) in enumerate(zip(file_list, y_pred)):
    print(f"音频 {filename} 预测的乐器是: {instrument_classes[pred]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
音频 1-cello.wav 预测的乐器是: cello
音频 10 clarinet.wav 预测的乐器是: clarinet
音频 11 electric guitar.wav 预测的乐器是: piano
音频 12 flute.wav 预测的乐器是: flute
音频 13 human vocie.wav 预测的乐器是: vocal
音频 14 human voice-1.wav 预测的乐器是: vocal
音频 15 human voice.wav 预测的乐器是: vocal
音频 16 orgen.wav 预测的乐器是: piano
音频 17 pinao.wav 预测的乐器是: piano
音频 17 saxphone.wav 预测的乐器是: clarinet
音频 19 trumpet.wav 预测的乐器是: trumpet
音频 2-acoustic guitar.wav 预测的乐器是: acoustic guitar
音频 20 violin.wav 预测的乐器是: violin
音频 3-oboe.wav 预测的乐器是: clarinet
音频 4-organ.wav 预测的乐器是: piano
音频 5-piano.wav 预测的乐器是: piano
音频 6-trumpet.wav 预测的乐器是: trumpet
音频 7-violin.wav 预测的乐器是: violin
音频 8 acoustic guitar.wav 预测的乐器是: acoustic guitar
音频 9 cello.wav 预测的乐器是: cello
